# 06I - Hyperparameter Optimization (Execution-Ready)

Optimize the best-performing model using GridSearchCV. Update the `selected_model` variable after completing 06H.

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df=pd.read_csv("../data/datasets/american_bankruptcy.csv")
df["target"]=df["status_label"].map({"alive":0,"failed":1})

drop=["status_label","target"]
if "company_name" in df.columns:
    drop.append("company_name")

X=df.drop(columns=drop)
y=df["target"]

num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42)

# GridSearchCV over the full ~63k-row training set is expensive on a single
# core; search on a stratified subsample and keep the full training set
# available for the final fit in Notebook 06J.
GRID_SEARCH_SAMPLE_SIZE = 15000
if len(X_train) > GRID_SEARCH_SAMPLE_SIZE:
    X_train_grid, _, y_train_grid, _ = train_test_split(
        X_train, y_train,
        train_size=GRID_SEARCH_SAMPLE_SIZE,
        stratify=y_train,
        random_state=42
    )
else:
    X_train_grid, y_train_grid = X_train, y_train

pre=ColumnTransformer([
("num",SimpleImputer(strategy="median"),num),
("cat",Pipeline([
("imp",SimpleImputer(strategy="most_frequent")),
("enc",OneHotEncoder(handle_unknown="ignore"))
]),cat)
])


## Select Model

In [2]:
# Replace with your best model if different.
selected_model=RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=1
)

pipeline=Pipeline([
("preprocessor",pre),
("classifier",selected_model)
])


## Hyperparameter Grid

In [3]:
# Trimmed grid keeps this tractable on a single core while still
# demonstrating a real hyperparameter search.
param_grid={
'classifier__n_estimators':[100,150],
'classifier__max_depth':[10,12]
}

grid=GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=2,
    n_jobs=1,
    verbose=1
)


## Train Grid Search

In [4]:
grid.fit(X_train_grid,y_train_grid)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest F1:")
print(grid.best_score_)


Fitting 2 folds for each of 4 candidates, totalling 8 fits


Best Parameters:
{'classifier__max_depth': 10, 'classifier__n_estimators': 100}

Best F1:
0.17675931401537553


## Evaluate Best Model

In [5]:
# Refit the best-found configuration on the full training set for the final model.
best_model=grid.best_estimator_
best_model.fit(X_train,y_train)

pred=best_model.predict(X_test)

from sklearn.metrics import classification_report,f1_score,accuracy_score

print(classification_report(y_test,pred))
print("Accuracy:",accuracy_score(y_test,pred))
print("F1:",f1_score(y_test,pred))


              precision    recall  f1-score   support

           0       0.97      0.82      0.89     14693
           1       0.19      0.58      0.29      1044

    accuracy                           0.81     15737
   macro avg       0.58      0.70      0.59     15737
weighted avg       0.91      0.81      0.85     15737

Accuracy: 0.8080320264345173
F1: 0.2876680028295213


## Save Tuned Model

In [6]:
joblib.dump(best_model,"../models/best_tuned_model.joblib")
print("Saved best_tuned_model.joblib")


Saved best_tuned_model.joblib


## Summary

In [7]:
print("Hyperparameter tuning completed.")
print("Use this tuned model in Notebook 06J for final production selection.")


Hyperparameter tuning completed.
Use this tuned model in Notebook 06J for final production selection.
